# KV Caching with Valkey

**Difficulty:** Intermediate · Python · ~20 min

Caching the attention KV that an LLM computes for a shared prompt prefix cuts cost
per request and shortens time-to-first-token. This walkthrough stores that KV
in Valkey and demonstrates how it can be reused across vLLM instances.

**By the end** you'll have two vLLM replicas sharing one KV cache in Valkey, and
you'll watch a cold request store its KV, a warm request read it back, and a
second replica reuse the first replica's cached prefix — all on CPU.

When an LLM answers a prompt, it first computes attention key/value tensors for
every input token. This is the prefill step: its cost is proportional to the
prompt length, and the result is normally thrown away after each request.
Whenever a later request shares a prefix with an earlier one, prefill recomputes
the same KV from scratch.

Caching that KV and reloading it skips the recompute: you pay for prefill once
and reuse it. Shared prefixes are common in practice:

- **Multi-turn chat:** each turn typically resends much of the conversation so
  far, so a later turn shares a long prefix with the earlier ones. How much is
  resent depends on the application, but something almost always is.
- **A common preamble:** a system prompt, tool and function definitions, MCP
  context, skills, or other instructions sit in front of many requests.
- **RAG and agentic memory:** retrieved passages or recalled memories are often
  reused across requests, depending on the retrieval strategy.

[LMCache](https://github.com/LMCache/LMCache) is the KV-cache layer that plugs
into vLLM and does this. It can store KV in several places, including local
memory, disk, or a remote backend; this walkthrough runs a standalone LMCache
server that uses Valkey as its backend. vLLM's own built-in prefix cache is limited to what one engine can hold
and is not shared with other engines. Valkey is a good fit when several vLLM
instances must read one cache, and that cache needs to be larger than any single
host can hold. The instances sharing a cache must run the same model and tensor
configuration, since the cached KV is specific to both.

## Running this notebook

This notebook runs against the live stack. Set up the stack and kernel first —
the [cookbook README](README.md) has the prerequisites and the one-time setup
(start the containers, create the `kv-caching-valkey` kernel).

With the stack healthy and the `kv-caching-valkey` kernel selected, run the code
cells top to bottom (Shift+Enter, or Run All). Each cell prints what to look for,
such as key counts and Valkey `GET` calls, so you can watch the cache fill up and
get reused. Run them in order, because later cells build on the conversation
state that earlier cells set up.


## How LMCache is pointed at Valkey

On CPU, vLLM has no in-worker KV connector, so each replica uses the
multi-process connector (`LMCacheMPConnector`) and hands its KV to a standalone
`lmcache server`. That server is what talks to Valkey, through its L2 adapter.
In [`docker-compose.yml`](docker-compose.yml) it is started with this command
(shown for reference, not to run):

```text
lmcache server --l1-size-gb 0.05 --eviction-policy noop \
  --l2-store-policy skip_l1 --supported-transfer-mode auto \
  --l2-adapter '{"type":"valkey","startup_nodes":"valkey:6379"}'
```

That `--l2-adapter` spec is what makes Valkey the backing store.
`--l2-store-policy skip_l1` keeps the server's L1 tier as a write buffer only,
so a cache *read* comes from Valkey rather than local memory — that is what
makes Valkey's role visible as `GET` calls later in this notebook. A production
deployment usually keeps L1 hot for lower-latency reads and lets Valkey back it
as the larger shared tier. [`lmcache_config.yaml`](lmcache_config.yaml)
documents the same spec and the options you are most likely to change. See the
[LMCache MP server configuration reference](https://docs.lmcache.ai/mp/configuration.html)
for the full option set.

> **Security:** This stack uses no authentication or TLS for simplicity.
> For any non-localhost deployment, enable authentication and TLS.
> See the [Valkey security documentation](https://valkey.io/topics/security/).

In [ ]:
# ── First cell to run (after the stack is up). Run the code cells top to
# ── bottom from here; each one prints what to look for.
# A tiny bit of shared setup: talk to each replica over its OpenAI-compatible
# HTTP API, and read the KV-cache state from Valkey with the Valkey GLIDE client
# (the same client family the LMCache server uses for its Valkey L2 backend).
from openai import OpenAI
from glide_sync import GlideClient, GlideClientConfiguration, NodeAddress

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

# The two vLLM replicas, each mapped to a host port by docker-compose.yml.
replica_a = OpenAI(base_url="http://localhost:8001/v1", api_key="not-needed")
replica_b = OpenAI(base_url="http://localhost:8002/v1", api_key="not-needed")

# Connect to the Valkey that docker-compose exposes on localhost. Reading these
# signals (DBSIZE and INFO commandstats) is how we watch LMCache's L2 tier.
valkey_client = GlideClient.create(
    GlideClientConfiguration(addresses=[NodeAddress("127.0.0.1", 6379)])
)


def dbsize():
    """Number of keys in Valkey. Each cached KV chunk is stored as keys here."""
    return valkey_client.custom_command(["DBSIZE"])


def valkey_get_calls():
    """Cumulative Valkey GET/MGET calls (from INFO commandstats).

    The LMCache server runs with --l2-store-policy skip_l1, so L1 is only a
    write buffer and a cache *read* has to come from Valkey. Reuse therefore
    shows up here as GET calls.
    """
    info = valkey_client.custom_command(["INFO", "commandstats"])
    if isinstance(info, bytes):
        info = info.decode()
    total = 0
    for line in info.splitlines():
        if line.startswith(("cmdstat_get:", "cmdstat_mget:")):
            for field in line.split(":", 1)[1].split(","):
                if field.startswith("calls="):
                    total += int(field.split("=", 1)[1])
    return total


print("Setup ready.")

Confirm the whole stack is live before going further: Valkey answers `PING`, and
both replicas list the model on `/v1/models`.

In [ ]:
print("valkey ping :", valkey_client.ping())
print("replica A   :", [m.id for m in replica_a.models.list().data])
# Replica B is only exercised in the cross-replica section below, so we do
# not probe it here — that keeps the single-replica path (one vLLM) working
# up to the one cell you are told to skip.

## Example: a multi-turn chat

We walk through one scenario start to finish, so you can map it onto your own
use. We hold a short multi-turn conversation with replica A and watch the KV
cache in Valkey between turns. Then we send the same conversation to replica B
to prove the cache is actually shared.

We keep the conversation in an explicit `messages` list and grow it turn by
turn. That is what makes each turn share the previous turn's prefix, the same
thing that happens in any real chat UI.

In [ ]:
import uuid

# Two things shape this system prompt:
#
# 1. A fresh per-run id. The KV cache lives in Valkey until the stack is torn
#    down, so a fixed prompt would already be cached on a second run and Turn 1
#    below would not be a genuine cold miss. The id keeps every run cold. It
#    sits in the system prompt, so every turn in this run still shares it as a
#    prefix, which is what lets Turn 2 and the second replica reuse the cache.
# 2. Enough content to fill a chunk. LMCache stores KV in fixed-size chunks
#    (256 tokens by default), so a one-line prompt would store nothing and the
#    cold turn would not grow the key count. The store-policy block below gives
#    the shared prefix enough tokens to land at least one chunk in Valkey.
SESSION_ID = uuid.uuid4().hex

SYSTEM_PROMPT = (
    f"[session {SESSION_ID}] "
    "You are a customer service assistant for ValkeyMart, an online store. You "
    "answer one customer's questions about the status of their orders. \n\n"
    "Since this is a simulation use these two answers, the first for even order numbers, the second for odd order numbers.\n\n"
    "Even order numbers: Your order is being processed and should ship within the next 24 hours.\n\n"
    "Odd order numbers: Your order shipped today, check your email for a shipping notice with a link to the shipper.\n\n"
    "When you've provided a response ask the customer if they need more help.\n\n"
    "Reference policies you should assume when answering:\n"
    "- Standard shipping takes 3 to 5 business days; express takes 1 to 2.\n"
    "- Orders can be cancelled or changed within 60 minutes of being placed.\n"
    "- Returns are accepted within 30 days of delivery with the original receipt.\n"
    "- Refunds post to the original payment method within 5 to 7 business days.\n"
    "- Tracking numbers are issued once an order leaves the fulfillment center.\n"
    "- Delivery windows are estimates and can shift with weather or carrier load.\n"
    "- International orders may incur customs duties charged by the destination.\n"
    "- Loyalty members get free returns and early access to seasonal sales.\n"
    "- Gift orders can include a note and hide the price on the packing slip.\n"
    "- Damaged items are replaced at no cost when reported within 48 hours.\n"
)

# The conversation so far. We append to this between turns.
messages = [{"role": "system", "content": SYSTEM_PROMPT}]


def ask(client, question, conversation):
    """Send the whole conversation + a new question, return the answer text."""
    conversation.append({"role": "user", "content": question})
    resp = client.chat.completions.create(
        model=MODEL, messages=conversation, max_tokens=32, temperature=0.0,
    )
    answer = resp.choices[0].message.content
    conversation.append({"role": "assistant", "content": answer})
    return answer

### Turn 1: cold

Nothing is cached yet. Replica A has to prefill the full prompt (system prompt +
first question), computing KV for every token from scratch. What gets cached is
exactly that KV: the attention key/value tensors for the prompt's tokens, split
into fixed-size chunks. As replica A prefills, LMCache hands those KV chunks to
the LMCache server, which writes them to Valkey.

We record Valkey's key count before and after. A cold turn should make it
**grow**: the freshly computed KV blocks have landed in Valkey. (The system
prompt carries a fresh per-run id, so this turn is genuinely cold even if you
have run the notebook before against the same Valkey.)

In [ ]:
before = dbsize()
answer1 = ask(replica_a, "What is the state of order 123456?", messages)
after = dbsize()

print("Q1 answer     :", answer1)
print("Valkey dbsize :", before, "->", after)
print("KV blocks stored in Valkey:", after - before)

### Turn 2: warm, shared prefix

Now the follow-up. The conversation we send is `system + Q1 + A1 + Q2`, so its
first three parts are byte-for-byte the prefix replica A just prefilled. Only
the new question at the end is new.

**How the hit works.** LMCache splits the token sequence into fixed-size chunks
(`chunk_size: 256`) and keys each chunk by a hash of the tokens in that chunk
and every chunk before it. The Valkey key is a pure function of
`(model, world_size, worker_id, chunk_hash, dtype)`
(`vllm@<model>@<world>@<worker>@<hash>` on the wire), so two requests that start
with the same tokens produce the identical key. A write by one is a hit for the
other. The match is on an exact token prefix: identical tokens from the
start, up to the point where the two requests diverge. Turn 2 shares every token
of the system prompt and first exchange, so those chunks are a hit. LMCache
loads their KV from Valkey and vLLM only prefills the handful of new tokens in
Q2.

Two signals here. The cached prefix is **not stored again**: `dbsize` does not
grow for the shared prefix (only the handful of new tokens in Q2 can add
anything). And the prefix's KV is **read back from Valkey**: because the server
runs with `--l2-store-policy skip_l1` (L1 is a write buffer only), the read
comes from Valkey as GET calls rather than from an in-memory tier. So LMCache
loads the prefix's KV from Valkey instead of vLLM recomputing it.

In [ ]:
gets_before = valkey_get_calls()
before = dbsize()
answer2 = ask(replica_a, "What about Order 654321?", messages)
after = dbsize()
gets_after = valkey_get_calls()

print("Q2 answer            :", answer2)
print("Valkey dbsize        :", before, "->", after)
print("new KV blocks stored :", after - before)
print("Valkey GET calls      :", gets_before, "->", gets_after)
print()
print("The shared prefix (system + Q1 + A1) was already cached, so it is a hit,")
print("not a re-store: dbsize barely moves. The GET calls are LMCache loading")
print("that prefix's KV back from Valkey instead of vLLM recomputing it.")


### A second replica hits the first replica's cache

Replica B has served no traffic and computed no KV of its own. We send it the
same conversation prefix replica A built up. The KV for that prefix lives in
Valkey, not inside replica A's process, so replica B finds it there and loads
it instead of prefilling from scratch.

The signal: replaying replica A's prefix on replica B **reads that KV back from
Valkey** (GET calls increment) and stores **no new keys** (`dbsize` is
unchanged), because the prefix's KV is already in the shared store. Replica B
reuses what replica A computed. An in-process cache on replica A would be
invisible to replica B; putting the cache in Valkey is what makes this
cross-replica reuse possible.

In [ ]:
# This is the only cell to skip when running a single replica (see "Running
# on a smaller machine" below). It confirms replica B is up, then replays
# replica A's exact conversation prefix (everything up to, but not
# including, replica A's last answer) as a fresh request to replica B.
print("replica B   :", [m.id for m in replica_b.models.list().data])
shared_prefix = messages[:-1]

gets_before = valkey_get_calls()
before = dbsize()
resp = replica_b.chat.completions.create(
    model=MODEL, messages=shared_prefix, max_tokens=32, temperature=0.0,
)
after = dbsize()
gets_after = valkey_get_calls()

print("Replica B answer     :", resp.choices[0].message.content)
print("Valkey dbsize        :", before, "->", after, "(unchanged == not re-stored)")
print("Valkey GET calls      :", gets_before, "->", gets_after, "(went up == read from Valkey)")
print()
print("Replica B served no traffic of its own, yet answered by reading the KV")
print("replica A wrote to Valkey — the cache is genuinely shared through Valkey.")


### Running on a smaller machine

Short on memory? See
[Running on a smaller machine](README.md#running-on-a-smaller-machine-one-replica)
in the README for how to start just one replica. In that mode, skip the
cross-replica cell above (the one that calls `replica_b`) — everything else in
this notebook runs the same.


### What it saves: cold vs. warm latency

The cache-hit signals above show that KV is reused. The first request is cold — vLLM prefills
every token. The second shares the whole prefix, so LMCache loads that KV from
Valkey and vLLM skips the prefill.

On CPU the absolute millisecond numbers vary from run to run, so treat the
direction as the point rather than the exact timings. Skipping prefill on
the shared prefix usually cuts the warm request's time to first token by more
than half — run the cell below to see what you get. The cell asks for a single
output token (`max_tokens=1`), so it times the prefill the cache skips plus the
first token, not the token-by-token decode of a full answer. Decode costs the
same cold or warm, and on some CPUs (for example float16 on arm64) it would
dominate a 32-token answer and hide the saving. The saving grows with prompt
length and model size. On a GPU serving a production-sized model over a long
shared prefix, skipping prefill is what this cache is built for.

In [ ]:
import time

# A fresh, long prefix unique to this run, so the first request is genuinely
# cold (nothing cached yet).
timing_prefix = [
    {"role": "system", "content": SYSTEM_PROMPT + " " + "Reference context. " * 80},
    {"role": "user", "content": "What is the status of order 123456?"},
]


def timed_request(client, conversation):
    # One output token: this times the prefill (what the cache skips) and the
    # first token, not the decode of an answer, which costs the same cold or warm.
    start = time.perf_counter()
    client.chat.completions.create(
        model=MODEL, messages=conversation, max_tokens=1, temperature=0.0,
    )
    return time.perf_counter() - start


cold_s = timed_request(replica_a, timing_prefix)   # prefill + store to Valkey
warm_s = timed_request(replica_a, timing_prefix)   # load KV from Valkey, skip prefill

print(f"cold (prefill)      : {cold_s * 1000:7.1f} ms")
print(f"warm (cache hit)    : {warm_s * 1000:7.1f} ms")
if warm_s < cold_s:
    print(f"saved               : {(cold_s - warm_s) * 1000:7.1f} ms "
          f"({(1 - warm_s / cold_s) * 100:.0f}% faster)")
else:
    print("(no speedup this run — CPU timings are noisy; try re-running, and see the note above)")


## Cleanup

Tear down all four containers, Valkey, the LMCache server, and both vLLM replicas:

```bash
docker compose down
```

**What persists:** the KV cache lives in Valkey (the LMCache server's L2 tier),
so it survives an individual vLLM replica or the LMCache server restarting. The cache is lost when
`docker compose down` removes the container, unless you mount a data volume and
enable append-only persistence to keep it across container recreation. The
`hf-cache` volume that holds the downloaded model weights does persist across
`down`/`up`, so later runs skip the re-download.

## What you learned

- **Why Valkey:** it holds the KV cache *outside* any single vLLM process, so
  one cache can be shared across many vLLM instances and can outlive any single
  one of them. Replica B reused a prefix that replica A had computed, something
  an in-process cache cannot do. (The cache is in-memory; it only survives a
  Valkey restart if you enable persistence.)
- **The multi-turn payoff:** each turn resends the prior turn's prefix, so the
  warm turn loaded that prefix's KV from Valkey instead of recomputing it.
- **The mechanism:** LMCache keys each KV chunk by a hash of its token prefix,
  so requests sharing an exact token prefix land on the same Valkey keys. A
  cold turn stores them, and later turns and other replicas read them back from
  Valkey. (This demo sets `--l2-store-policy skip_l1` so reads come straight
  from Valkey and its role is visible; a production deployment usually keeps the
  L1 tier hot for lower-latency reads.)

## From this demo to a real assistant

The system prompt here hard-codes two canned answers (one for even order
numbers, one for odd) so the walkthrough stays deterministic and offline. In a
real deployment the assistant would not invent order status this way: it would
call your order system for the actual state of the order, answer from that, and
offer the customer follow-up options such as more detail or looking up another
order. The caching mechanics shown here are unchanged by that — whatever shapes
the shared prompt prefix, LMCache still keys its KV chunks on that prefix and
reuses them across turns and replicas through Valkey.


## Next steps

- **Scale out.** The natural continuation of the two-replica step: run *more*
  vLLM replicas, all sharing this one Valkey. Every replica warms the same cache
  and reads every other replica's prefixes, so the hit rate climbs as you add
  replicas. (This is scaling the vLLM fleet against one Valkey, not adding Valkey
  replicas.)
- **Take it to production.** See the
  [LMCache production deployment guide](https://docs.lmcache.ai/production/docker_deployment.html)
  for running LMCache under a served vLLM, and the
  [LMCache Valkey/Redis backend docs](https://docs.lmcache.ai/kv_cache/redis.html)
  for cluster mode, high availability, and tuning the remote store.